In [7]:
import json
import sys
from rich import print as rprint
from pathlib import Path
import re

nb_dir = Path.cwd()

project_root = nb_dir.parent.parent

sys.path.insert(0, str(project_root))

from scripts.text_matching import normalise_text,remove_diacritics

In [8]:
people_file = Path(project_root / "data/from db/people.json")
variants_file = Path(project_root / "data/from db/people_variants.json")
unmatched_file = Path(project_root / "data/people/unmatched_flat.json")
nopes_file = Path(project_root / "scripts/notebooks/nopes_fixed.json")

In [9]:
with open(people_file , "r") as f:
   people = json.load(f)
with open(variants_file , "r") as f:
   variants = json.load(f)
with open(unmatched_file, "r") as f:
   unmatched = json.load(f)
with open(nopes_file, "r") as f:
   nopes = json.load(f)


In [10]:
unmatched_dict = {}
# rprint(f"unmatched entries: {len(unmatched)}")
total_unmatched_flat = len(unmatched)
total_nopes = len(nopes)

for entry in unmatched:
    composite_id = entry["composite_id"]

    if composite_id not in unmatched_dict:
        unmatched_dict[composite_id] = [entry]
    else:
        unmatched_dict[composite_id].append(entry)

rprint(f"people total: {len(people)}")
rprint(f"nopes: {len(nopes)}")


# rprint(dict(list(unmatched_dict.items())[:1]))
# rprint(f"len lookup: {len(unmatched_dict)}")


people total: 7817

nopes: 1223

In [11]:
new_people_complete = {}
people_new = []
b2p_new = []

single_count = 0
single_check_count = 0
singles_to_check = {}
nope_processed_count = 0
composite_id_processed = 0


for name, entry in nopes.items():
# for name, entry in list(nopes.items())[:100]:
    nope_processed_count += 1
    last_norm = entry["last_norm"]
    first_norm = entry["first_norm"]
    single_norm = entry["single_norm"]
    family_name = entry["family_name"]
    given_names = entry["given_names"]
    single_name = entry["single_name"]
    name_particles = entry["name_particles"]

    # CREATE UNIFIED ID
    if single_norm:
        single_count += 1
        if single_norm.isalpha():
            unified_id_base = single_norm
            continue

        if " " in single_norm:
            single_strip = single_norm.replace(",", "")
            single_strip.replace("/", "")
            single_split = single_strip.split(" ")
            if len(single_split) == 2:
                single_first_2 = single_split[0] + "_" + single_split[1]
                unified_id_base = single_first_2
            else:
                single_first_2 = single_split[0] + "_" + single_split[1]
                initials = "_".join(i[0] for i in single_split[2:])
                single_unified = single_first_2 + "_" + initials
                unified_id_base = single_unified

        if family_name or first_norm:
            single_check_count += 1
            singles_to_check[name] = entry

    elif family_name and not first_norm:
        single_check_count += 1
        singles_to_check[name] = entry

    else:
        if first_norm.isalpha():
            first_unified = first_norm
            # rprint(f"first alpha: {first_unified}")
            continue
        else:
            if " " in first_norm:
                first_strip = first_norm.replace(".", "")
                # rprint(f"stripped: {first_strip}")
                first_split = first_strip.split(" ")
                # rprint(f"split: {first_split}")
                first_full = first_split[0]
                initials = "_".join(n[0] for n in first_split[1:])
                # rprint(f"initials: {initials}")
                first_unified = first_full + "_" + initials
                # rprint(f"first with stuff: {first_unified}")
        unified_id_base = last_norm + "_" + first_unified

    unified_id = remove_diacritics(unified_id_base)
    # rprint(f"unified, no diacritics: {unified_id}")

    # FORMAT NAMES

    if family_name and family_name.isupper():
        family_formatted = family_name.title()
        family_name = family_formatted

    if given_names and given_names.isupper():
        given_f = given_names.title()
        given_names = given_f

    if single_name and single_name.isupper():
        single_f = single_name.title()
        single_name = single_f


    new_people_complete[unified_id] = {
        "unified_id": unified_id,
        "family_name": family_name,
        "given_names": given_names,
        "name_particles": name_particles,
        "single_name": single_name,
        "is_organisation": entry["is_organisation"],
        "entries": []
    }
    # rprint(f"1: people complete, no entries: {new_people_complete}")
    new_people_complete_sorted = dict(sorted(new_people_complete.items()))

    people_new.append({
        "unified_id": unified_id,
        "family_name": family_name,
        "given_names": given_names,
        "name_particles": name_particles,
        "single_name": single_name,
        "is_organisation": entry["is_organisation"],
    })
    # rprint(f"2: people list: {people_new}")


    for composite_id in entry["composite_id"]:
        composite_id_processed += 1
        records = unmatched_dict[composite_id]
        for u in records:
            if u["display_norm"] == name:
                display_norm = u["display_norm"]
                display_name = u["display_name"]
                sort_order = u["sort_order"]
                is_author = u["is_author"]
                is_editor = u["is_editor"]
                is_contributor = u["is_contributor"]
                is_translator = u["is_translator"]

                new_people_complete_sorted[unified_id]["entries"].append({
                    "display_name": display_name,
                    "composite_id": composite_id,
                    "sort_order": sort_order,
                    "is_author": is_author,
                    "is_editor": is_editor,
                    "is_contributor": is_contributor,
                    "is_translator": is_translator,
                })
                # rprint(f"3: people complete with entries: {new_people_complete}")
                # break

                b2p_new.append({
                    "composite_id": composite_id,
                    "unified_id": unified_id,
                    "display_name": display_name,
                    "family_name": family_name,
                    "given_names": given_names,
                    "name_particles": name_particles,
                    "single_name": single_name,
                    "sort_order": sort_order,
                    "is_author": is_author,
                    "is_editor": is_editor,
                    "is_contributor": is_contributor,
                    "is_translator": is_translator,
                })
                # rprint(f"4: b2p info: {b2p_new}")
                break

if single_check_count > 0:
    with open("singles_to_check.json", "w") as f:
        json.dump(singles_to_check, f, ensure_ascii=False, indent=2)

with open("people_new_complete.json", "w") as f:
    json.dump(new_people_complete_sorted, f, ensure_ascii=False, indent=2)

with open("people_new.json", "w") as f:
    json.dump(people_new, f, ensure_ascii=False, indent=2)
with open("b2p_new.json", "w") as f:
    json.dump(b2p_new, f, ensure_ascii=False, indent=2)


rprint(f"""
    === LOG ===

    total unmatched: {total_unmatched_flat}
    people entries in nopes: {total_nopes}
    singles found: {single_count}
    singles_to_check: {single_check_count}
    people processed: {nope_processed_count}
    new_people_complete count: {len(new_people_complete)}
    people_new count: {len(people_new)}
    composite_ids processed: {composite_id_processed}
    b2p new count: {len(b2p_new)}

    === DONE ===
""")


=== LOG ===

    total unmatched: 1860
    people entries in nopes: 1223
    singles found: 75
    singles_to_check: 0
    people processed: 1223
    new_people_complete count: 301
    people_new count: 305
    composite_ids processed: 324
    b2p new count: 283

    === DONE ===